<a href="https://colab.research.google.com/github/sudhars97/Ecommerce-taxonomy-pipeline/blob/main/notebooks/02_llm_taxonomy_enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required libraries
!pip install langchain langchain-core langchain-google-genai pandas -q

import os
import pandas as pd
import json
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# 2. Securely load your API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 3. Initialize the LLM (Removed temperature to fix the warning)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# 4. Load the messy data
print("Loading messy catalog data...")
messy_data = {
    'product_id': ['B07R4PHDWW', 'B000FAJEZM', 'B07XT2LV64', 'B07PCVN39G', 'B01M0SGFV0'],
    'product_title': [
        "EunWow Anchor Pirate Black Chain Rocker Cool Necklace",
        "Zipper Rescue Zipper Repair Kits – The Original",
        "Spartan Industrial - 1.5” X 1.5” (1000 Count) Reclosable",
        "GILLRAJ Price Tags with Strings Attached Pack",
        "Generic Mens Summer Casual Shorts"
    ],
    'product_description': ["Anchor Pendant", "EBC", "1.5\" x 1.5\"", "8966000029927", "shorts"],
    'product_brand': ["", "", "Spartan Industrial", "", ""]
}
messy_products = pd.DataFrame(messy_data)

# 5. The Few-Shot Prompt Template
taxonomy_prompt = PromptTemplate.from_template("""
You are an expert eCommerce Taxonomy AI for a major marketplace.
Your task is to read a messy product title and extract structured data.
If a detail is missing, guess it from the title, or return "Unknown".
Always respond in perfectly valid JSON.

Here are examples of how to do this:

Title: "Nike Men's Air Max 270 Running Shoe Black/White Size 10"
JSON: {{"brand": "Nike", "product_category": "Footwear", "target_gender": "Men", "color": "Black/White"}}

Title: "Spartan Industrial - 1.5” X 1.5” (1000 Count) Reclosable Zip Lock Bags"
JSON: {{"brand": "Spartan Industrial", "product_category": "Packaging Bags", "target_gender": "Unisex", "color": "Clear"}}

Now, enrich this product:
Title: "{product_title}"
JSON:
""")

# 6. Run the LLM Enrichment
print("\nStarting LLM Enrichment Pipeline...")
print("-" * 50)

enriched_catalog = []

for index, row in messy_products.iterrows():
    title = row['product_title']
    print(f"RAW TITLE: {title}")

    # Format the prompt and ask the LLM
    formatted_prompt = taxonomy_prompt.format(product_title=title)
    response = llm.invoke(formatted_prompt)

    # --- FIX: Safely extract text whether it returns a string or a list ---
    raw_content = response.content
    if isinstance(raw_content, list):
        content_str = raw_content[0].get('text', '') if isinstance(raw_content[0], dict) else str(raw_content[0])
    else:
        content_str = str(raw_content)

    # Clean the JSON output
    clean_json = content_str.replace("```json", "").replace("```", "").strip()
    print(f"ENRICHED TAXONOMY: {clean_json}")
    print("-" * 50)

    # Save the data
    try:
        parsed_data = json.loads(clean_json)
        parsed_data['original_id'] = row['product_id']
        enriched_catalog.append(parsed_data)
    except Exception as e:
        print("Failed to parse JSON")

# 7. Display the final structured DataFrame
enriched_df = pd.DataFrame(enriched_catalog)
print("\nFinal Structured DataFrame:")
display(enriched_df)

Loading messy catalog data...

Starting LLM Enrichment Pipeline...
--------------------------------------------------
RAW TITLE: EunWow Anchor Pirate Black Chain Rocker Cool Necklace
ENRICHED TAXONOMY: {"brand": "EunWow", "product_category": "Jewelry", "target_gender": "Unisex", "color": "Black"}
--------------------------------------------------
RAW TITLE: Zipper Rescue Zipper Repair Kits – The Original
ENRICHED TAXONOMY: {"brand": "Zipper Rescue", "product_category": "Zipper Repair Kits", "target_gender": "Unisex", "color": "Unknown"}
--------------------------------------------------
RAW TITLE: Spartan Industrial - 1.5” X 1.5” (1000 Count) Reclosable
ENRICHED TAXONOMY: {"brand": "Spartan Industrial", "product_category": "Packaging Bags", "target_gender": "Unisex", "color": "Clear"}
--------------------------------------------------
RAW TITLE: GILLRAJ Price Tags with Strings Attached Pack
ENRICHED TAXONOMY: {"brand": "GILLRAJ", "product_category": "Price Tags", "target_gender": "Unis

,brand,product_category,target_gender,color,original_id
0,EunWow,Jewelry,Unisex,Black,B07R4PHDWW
1,Zipper Rescue,Zipper Repair Kits,Unisex,Unknown,B000FAJEZM
2,Spartan Industrial,Packaging Bags,Unisex,Clear,B07XT2LV64
3,GILLRAJ,Price Tags,Unisex,White,B07PCVN39G
4,Generic,Shorts,Men,Unknown,B01M0SGFV0
